In [ ]:
import torch

if torch.cuda.is_available():
    print(f"CUDA is available. Using GPU.")

    # Number of CUDA devices
    num_gpus = torch.cuda.device_count()
    print(f"Number of devices: {num_gpus}")

    # Information for each available GPU
    for i in range(num_gpus):
        print(f"--- Device {i} ---")
        print(f"Device name: {torch.cuda.get_device_name(i)}")

        # Get device properties
        props = torch.cuda.get_device_properties(i)
        print(f"Total memory: {round(props.total_memory / (1024**3), 2)} GB")
        print(f"CUDA Capability: {props.major}.{props.minor}")

        # Current memory usage for the selected device (requires a tensor/model to be on the GPU)
        # Note: These values might be 0 if no tensors have been moved to the GPU yet.
        allocated_memory = round(torch.cuda.memory_allocated(i) / (1024**3), 2)
        reserved_memory = round(torch.cuda.memory_reserved(i) / (1024**3), 2)
        print(f"Allocated memory: {allocated_memory} GB")
        print(f"Reserved memory: {reserved_memory} GB")

    # Current active device
    print(f"Current active device: {torch.cuda.current_device()}")

else:
    print("CUDA is not available. Using CPU.")


CUDA is available. Using GPU.
Number of devices: 1
--- Device 0 ---
Device name: NVIDIA A100-SXM4-40GB
Total memory: 39.56 GB
CUDA Capability: 8.0
Allocated memory: 0.0 GB
Reserved memory: 0.0 GB
Current active device: 0


In [ ]:
import argparse
import json
from pathlib import Path
from typing import Dict, Any, List, Optional

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from trl import DPOTrainer, DPOConfig

In [ ]:
def load_preferences(path: Path) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Combined preferences file must be a JSON array")
    return data


def build_prompt(question: str, policy_name: Optional[str], context: str) -> str:
    """Build an instruction-style prompt for Mistral and include full document context."""
    return (
        "You are an expert policy analyst tasked with answering questions about AI policy and regulations. "
        "Provide direct, factual information grounded in the context. "
        "Cite relevant sources or document IDs where applicable, but do NOT use or generate external links. "
        "If the context does not contain enough information, state that explicitly instead of speculating. "
        f"Context: {context}\n\n"
        f"Question: {question}\n\n"
        "Provide: a comprehensive answer with a direct answer to the question, and citations to relevant sources where necessary. "
        "Do not include URLs or external links in your answer. "
        "Answer: "
    )

In [ ]:
def load_document_context(doc_id: Optional[int], fulltext_dir: Path) -> str:
    if doc_id is None:
        return "(No document_id provided.)"

    doc_path = fulltext_dir / f"{doc_id}.txt"
    if not doc_path.exists():
        return f"(Context file not found: {doc_path.name})"

    try:
        with open(doc_path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    except Exception as e:
        return f"(Error reading context: {e})"


In [ ]:
def make_dpo_dataset(rows: List[Dict[str, Any]], fulltext_dir: Path) -> Dataset:
    prompts, chosen, rejected = [], [], []

    for r in rows:
        q = r.get("question", "")
        policy_name = r.get("policy_name")
        a1 = r.get("answer_1", "").strip()
        a2 = r.get("answer_2", "").strip()
        pref = r.get("preferred")
        doc_id = r.get("document_id")

        if pref not in (1, 2) or not a1 or not a2:
            continue

        context = load_document_context(doc_id if isinstance(doc_id, int) else None, fulltext_dir)
        prompt = build_prompt(q, policy_name, context)

        c = a1 if pref == 1 else a2
        rj = a2 if pref == 1 else a1

        prompts.append(prompt)
        chosen.append(c)
        rejected.append(rj)

    if not prompts:
        raise ValueError("No valid preference rows found")

    return Dataset.from_dict({"prompt": prompts, "chosen": chosen, "rejected": rejected})


In [ ]:
prefs_path = Path("dpo_preferences_combined.json")
fulltext_dir = Path("fulltext")

rows = load_preferences(prefs_path)
dset = make_dpo_dataset(rows, fulltext_dir)

len(dset)

2000

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
training_args = DPOConfig(
    output_dir="dpo_mistral_ckpt",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=5e-6,

    # Mixed precision automatically handled for 8-bit
    fp16=False,
    bf16=True,                       # safe for most GPUs

    optim="paged_adamw_8bit",        # recommended for 8-bit training
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to=["none"],
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # works for Llama models
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 7,248,547,840 || trainable%: 0.0940


In [ ]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,   # TRL clones automatically
    args=training_args,
    train_dataset=dset,
)

Extracting prompt in train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
10,0.704400
20,0.705500
30,0.679000
40,0.655000
50,0.658500
60,0.666800
70,0.629500
80,0.625400
90,0.615200
100,0.628900


TrainOutput(global_step=125, training_loss=0.6483970260620118, metrics={'train_runtime': 4372.7094, 'train_samples_per_second': 0.457, 'train_steps_per_second': 0.029, 'total_flos': 0.0, 'train_loss': 0.6483970260620118, 'epoch': 1.0})

In [ ]:
trainer.save_model("dpo_mistral_ckpt")
tokenizer.save_pretrained("dpo_mistral_ckpt")

print("Training complete.")

Training complete.


In [ ]:
!zip -r dpo_mistral_ckpt.zip dpo_mistral_ckpt

  adding: dpo_mistral_ckpt/ (stored 0%)
  adding: dpo_mistral_ckpt/checkpoint-125/ (stored 0%)
  adding: dpo_mistral_ckpt/checkpoint-125/tokenizer.model (deflated 55%)
  adding: dpo_mistral_ckpt/checkpoint-125/chat_template.jinja (deflated 64%)
  adding: dpo_mistral_ckpt/checkpoint-125/rng_state.pth (deflated 26%)
  adding: dpo_mistral_ckpt/checkpoint-125/training_args.bin (deflated 54%)
  adding: dpo_mistral_ckpt/checkpoint-125/optimizer.pt (deflated 12%)
  adding: dpo_mistral_ckpt/checkpoint-125/scheduler.pt (deflated 61%)
  adding: dpo_mistral_ckpt/checkpoint-125/trainer_state.json (deflated 72%)
  adding: dpo_mistral_ckpt/checkpoint-125/special_tokens_map.json (deflated 74%)
  adding: dpo_mistral_ckpt/checkpoint-125/tokenizer.json (deflated 85%)
  adding: dpo_mistral_ckpt/checkpoint-125/adapter_model.safetensors (deflated 7%)
  adding: dpo_mistral_ckpt/checkpoint-125/tokenizer_config.json (deflated 68%)
  adding: dpo_mistral_ckpt/checkpoint-125/README.md (deflated 65%)
  adding: dp